In [1]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:


from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

EO_FILE = "EO_data.csv"
DEFAULT_TOLERANCE = pd.Timedelta(minutes=30)

EO_REQUIRED_COLUMNS = {"datetime", "EO_radiation", "station"}
WEATHER_REQUIRED_COLUMNS = {"timestamp", "station"}


# ---------------------------------------------------------------------------
#  EO loading and quality analysis
# ---------------------------------------------------------------------------

def load_eo_data(filepath=EO_FILE):
    """Load the EO dataset generated by PB-06."""
    filepath = Path(filepath)

    if not filepath.exists():
        raise FileNotFoundError(
            f"EO file not found: {filepath}. "
            "Run the EO download/parsing part (PB-06) first."
        )

    eo = pd.read_csv(filepath)

    missing = EO_REQUIRED_COLUMNS - set(eo.columns)
    if missing:
        raise ValueError(
            f"EO dataset is missing required columns: {sorted(missing)}"
        )

    return eo


def inspect_eo_data(eo):
    """
    Return a compact data-quality report.

    This is intentionally an inspection step: zero radiation is NOT treated
    as missing because nighttime solar radiation can legitimately be zero.
    """
    report = {
        "rows": len(eo),
        "columns": len(eo.columns),
        "stations": eo["station"].nunique(),
        "duplicate_rows": int(eo.duplicated().sum()),
        "missing_datetime": int(eo["datetime"].isna().sum()),
        "missing_radiation": int(eo["EO_radiation"].isna().sum()),
        "missing_station": int(eo["station"].isna().sum()),
    }

    return pd.Series(report, name="EO data quality")


def detect_eo_outliers(eo):
    """
    Detect radiation outliers using the IQR method.

    Outliers are flagged rather than automatically deleted because unusually
    high solar radiation values can be real observations.
    """
    values = pd.to_numeric(eo["EO_radiation"], errors="coerce")

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (values < lower_bound) | (values > upper_bound)

    return {
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": int(outlier_mask.sum()),
        "outlier_percentage": float(outlier_mask.mean() * 100),
        "mask": outlier_mask,
    }


def preprocess_eo_data(eo, interpolation_limit=4):
    """
    Clean and standardize EO observations.

    Steps:
        1. Convert datetime to pandas datetime.
        2. Convert EO radiation to numeric.
        3. Remove invalid timestamps.
        4. Replace +/- infinity with NaN.
        5. Remove duplicate station/timestamp observations.
        6. Interpolate short gaps within each station.
        7. Use station median only for values that remain missing.
        8. Keep the result sorted by station and timestamp.

    A 4-step interpolation limit is conservative and matches the idea of
    filling short gaps without inventing long missing periods.
    """
    eo = eo.copy()

    # Datetime conversion: EO keys are YYYYMMDDHH.
    eo["datetime"] = pd.to_datetime(
        eo["datetime"].astype(str),
        format="%Y%m%d%H",
        errors="coerce",
    )

    eo["EO_radiation"] = pd.to_numeric(
        eo["EO_radiation"],
        errors="coerce",
    )

    # Invalid numerical values should be treated as missing.
    eo["EO_radiation"] = eo["EO_radiation"].replace(
        [np.inf, -np.inf],
        np.nan,
    )

    # Remove rows that cannot participate in a station/time join.
    eo = eo.dropna(subset=["datetime", "station"])

    # Preserve one observation per station/timestamp.
    eo = (
        eo.sort_values(["station", "datetime"])
        .drop_duplicates(
            subset=["station", "datetime"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    # Track missing values before filling.
    eo["EO_was_missing"] = eo["EO_radiation"].isna()

    # Short-gap interpolation per station.
    eo = eo.sort_values(["station", "datetime"])

    eo["EO_radiation"] = (
        eo.groupby("station", group_keys=False)["EO_radiation"]
        .apply(
            lambda s: s.interpolate(
                method="linear",
                limit=interpolation_limit,
                limit_direction="both",
            )
        )
        .reset_index(level=0, drop=True)
    )

    # Remaining missing values: station-level median fallback.
    station_medians = eo.groupby("station")["EO_radiation"].transform("median")
    eo["EO_radiation"] = eo["EO_radiation"].fillna(station_medians)

    # Final fallback is only used if a complete station has no valid values.
    eo["EO_radiation"] = eo["EO_radiation"].fillna(0)

    eo = eo.sort_values(["station", "datetime"]).reset_index(drop=True)

    return eo


# ---------------------------------------------------------------------------
#   Visualizations / EDA
# ---------------------------------------------------------------------------

def plot_eo_distribution(eo, bins=50):
    """Plot the distribution of EO radiation values."""
    plt.figure(figsize=(10, 5))
    plt.hist(eo["EO_radiation"].dropna(), bins=bins)
    plt.xlabel("EO Radiation")
    plt.ylabel("Frequency")
    plt.title("EO Radiation Distribution")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_eo_boxplot(eo):
    """Boxplot used for IQR-based outlier inspection."""
    plt.figure(figsize=(10, 4))
    plt.boxplot(eo["EO_radiation"].dropna(), vert=False)
    plt.xlabel("EO Radiation")
    plt.title("EO Radiation - IQR Outlier Inspection")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_station_radiation(eo, top_n=10):
    """
    Compare mean EO radiation across stations.

    The chart is intentionally limited to the stations with the highest
    mean radiation so it remains readable with 40 stations.
    """
    station_mean = (
        eo.groupby("station")["EO_radiation"]
        .mean()
        .sort_values(ascending=False)
        .head(top_n)
    )

    plt.figure(figsize=(11, 5))
    station_mean.plot(kind="bar")
    plt.xlabel("Station")
    plt.ylabel("Mean EO Radiation")
    plt.title(f"Top {top_n} Stations by Mean EO Radiation")
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_eo_timeseries(eo, station=None, start=None, end=None):
    """
    Plot EO radiation over time for one station.

    If station is not provided, the first station in sorted order is used.
    """
    if station is None:
        station = sorted(eo["station"].dropna().unique())[0]

    data = eo[eo["station"] == station].copy()

    if start is not None:
        data = data[data["datetime"] >= pd.to_datetime(start)]

    if end is not None:
        data = data[data["datetime"] <= pd.to_datetime(end)]

    data = data.sort_values("datetime")

    plt.figure(figsize=(12, 5))
    plt.plot(data["datetime"], data["EO_radiation"], linewidth=1)
    plt.xlabel("Time")
    plt.ylabel("EO Radiation")
    plt.title(f"EO Radiation Over Time - {station}")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_missing_values(eo):
    """Visualize missing EO values before preprocessing."""
    missing = eo.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    plt.figure(figsize=(9, 4))

    if missing.empty:
        plt.text(
            0.5,
            0.5,
            "No missing values detected",
            ha="center",
            va="center",
            fontsize=14,
        )
        plt.axis("off")
    else:
        missing.plot(kind="bar")
        plt.xlabel("Column")
        plt.ylabel("Missing values")
        plt.title("EO Missing-Value Profile")
        plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------------------------
#  EO + Weather integration
# ---------------------------------------------------------------------------

def prepare_weather_for_integration(weather_df):
    """Validate and standardize the weather dataframe for the join."""
    weather = weather_df.copy()

    missing = WEATHER_REQUIRED_COLUMNS - set(weather.columns)
    if missing:
        raise ValueError(
            f"Weather dataframe is missing required columns: {sorted(missing)}"
        )

    weather["timestamp"] = pd.to_datetime(
        weather["timestamp"],
        errors="coerce",
    )

    weather = weather.dropna(
        subset=["timestamp", "station"]
    ).copy()

    return weather


def integrate_eo_with_weather(
    weather_df,
    eo_df,
    tolerance=DEFAULT_TOLERANCE,
):
    """
    Integrate hourly EO radiation into the weather observations.

    Because weather measurements are more frequent than the hourly EO data,
    merge_asof() matches each weather observation to the nearest EO value
    from the same station, within the specified tolerance.

    This avoids the common mistake of using an exact timestamp merge and
    losing valid matches simply because one dataset is hourly and the other
    is 15-minute based.
    """
    weather = prepare_weather_for_integration(weather_df)
    eo = eo_df.copy()

    eo["datetime"] = pd.to_datetime(eo["datetime"], errors="coerce")

    eo = eo.dropna(
        subset=["datetime", "station"]
    ).copy()

    # Keep only the columns needed for integration.
    eo_for_merge = eo[
        ["station", "datetime", "EO_radiation"]
    ].copy()

    # merge_asof requires sorting by the time key.
    weather = weather.sort_values(
        ["timestamp", "station"]
    ).reset_index(drop=True)

    eo_for_merge = eo_for_merge.sort_values(
        ["datetime", "station"]
    ).reset_index(drop=True)

    integrated = pd.merge_asof(
        weather,
        eo_for_merge,
        left_on="timestamp",
        right_on="datetime",
        by="station",
        direction="nearest",
        tolerance=tolerance,
    )

    # The EO timestamp is useful for validation, but the project's main
    # temporal key remains "timestamp".
    integrated = integrated.rename(
        columns={"datetime": "EO_timestamp"}
    )

    integrated["EO_match_found"] = integrated["EO_radiation"].notna()

    return integrated


def integration_report(integrated_df):
    """Return validation metrics for the EO/weather integration."""
    total_rows = len(integrated_df)

    if total_rows == 0:
        return pd.Series(
            {
                "integrated_rows": 0,
                "EO_matches": 0,
                "EO_missing_after_merge": 0,
                "EO_match_rate_%": 0.0,
                "duplicate_station_timestamp": 0,
            },
            name="Integration report",
        )

    duplicate_count = int(
        integrated_df.duplicated(
            subset=["station", "timestamp"]
        ).sum()
    )

    matched = int(integrated_df["EO_match_found"].sum())

    return pd.Series(
        {
            "integrated_rows": total_rows,
            "EO_matches": matched,
            "EO_missing_after_merge": total_rows - matched,
            "EO_match_rate_%": round(
                matched / total_rows * 100,
                2,
            ),
            "duplicate_station_timestamp": duplicate_count,
        },
        name="Integration report",
    )


def plot_integration_coverage(integrated_df):
    """Visualize successful vs unsuccessful EO matches."""
    matched = integrated_df["EO_match_found"].value_counts()

    labels = ["Matched", "Missing"]
    values = [
        int(matched.get(True, 0)),
        int(matched.get(False, 0)),
    ]

    plt.figure(figsize=(7, 4))
    plt.bar(labels, values)
    plt.ylabel("Number of weather observations")
    plt.title("EO Integration Coverage")
    plt.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_weather_vs_eo(integrated_df):
    """
    Scatter plot of weather radiation vs EO radiation.

    This is an EDA visualization only; it does not perform ML modeling.
    """
    required = {"radiation (W/m2)", "EO_radiation"}

    if not required.issubset(integrated_df.columns):
        print(
            "Skipping weather-vs-EO plot: required weather radiation "
            "column is not available."
        )
        return

    data = integrated_df[
        ["radiation (W/m2)", "EO_radiation"]
    ].dropna()

    if data.empty:
        print("Skipping weather-vs-EO plot: no matched observations.")
        return

    # Sampling keeps the visualization responsive on the full dataset.
    if len(data) > 10000:
        data = data.sample(10000, random_state=42)

    plt.figure(figsize=(8, 6))
    plt.scatter(
        data["radiation (W/m2)"],
        data["EO_radiation"],
        alpha=0.25,
        s=10,
    )
    plt.xlabel("Weather Radiation (W/m2)")
    plt.ylabel("EO Radiation")
    plt.title("Weather Radiation vs EO Radiation")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------------------------
# End-to-end Member 4 pipeline
# ---------------------------------------------------------------------------

def run_member4_pipeline(weather_df, eo_filepath=EO_FILE):
    """
    Run the complete Member 4 workflow.

    Returns:
        eo_clean: cleaned EO dataframe
        integrated: weather dataframe with EO features
        report: integration validation report
    """
    print("=" * 70)
    print("MEMBER 4 - EO PREPROCESSING & INTEGRATION")
    print("=" * 70)

    # -------------------- PB-07 --------------------
    print("\n[PB-07] Loading EO data...")
    eo_raw = load_eo_data(eo_filepath)

    print("\nRaw EO quality:")
    print(inspect_eo_data(eo_raw))

    print("\n[PB-07] Detecting potential outliers...")
    outlier_info = detect_eo_outliers(eo_raw)
    print(
        f"IQR bounds: "
        f"{outlier_info['lower_bound']:.2f} to "
        f"{outlier_info['upper_bound']:.2f}"
    )
    print(
        f"Potential outliers: {outlier_info['outlier_count']} "
        f"({outlier_info['outlier_percentage']:.2f}%)"
    )

    print("\n[PB-07] Preprocessing EO data...")
    eo_clean = preprocess_eo_data(eo_raw)

    print("\nClean EO shape:", eo_clean.shape)
    print("Stations:", eo_clean["station"].nunique())
    print(
        "Remaining EO missing values:",
        int(eo_clean["EO_radiation"].isna().sum()),
    )

    # -------------------- PB-08 --------------------
    print("\n[PB-08] Integrating EO with weather data...")
    integrated = integrate_eo_with_weather(
        weather_df,
        eo_clean,
    )

    report = integration_report(integrated)

    print("\nIntegration report:")
    print(report)

    print("\nFinal integrated shape:", integrated.shape)
    print("\nSample:")
    print(integrated.head())

    return eo_clean, integrated, report




In [4]:
import pandas as pd

WEATHER_PATH = r"C:\Users\josam\OneDrive\Desktop\TAHMO-Solar-Radiation-Prediction\TAHMO-Solar-Radiation-Prediction\Train_modified_V1.csv"
EO_PATH = r"C:\Users\josam\OneDrive\Desktop\TAHMO-Solar-Radiation-Prediction\TAHMO-Solar-Radiation-Prediction\EO_data.csv"  # adjust if EO_data.csv is elsewhere

weather_df = pd.read_csv(WEATHER_PATH)

print(weather_df.columns.tolist())
print(weather_df[["timestamp", "station"]].head())

# Run the pipeline
eo_clean, integrated, report = run_member4_pipeline(weather_df, eo_filepath=EO_PATH)

# Save the results next to the weather file
out_dir = r"C:\Users\josam\OneDrive\Desktop\TAHMO-Solar-Radiation-Prediction\TAHMO-Solar-Radiation-Prediction"
eo_clean.to_csv(out_dir + r"\EO_clean.csv", index=False)
integrated.to_csv(out_dir + r"\weather_EO_integrated.csv", index=False)
report.to_csv(out_dir + r"\integration_report.csv", header=True)

['ID', 'timestamp', 'precipitation (mm)', 'radiation (W/m2)', 'relativehumidity (-)', 'temperature (degrees Celsius)', 'station', 'station_name', 'country', 'installation_height', 'elevation', 'latitude', 'longitude', 'hour', 'minute', 'month', 'day_of_year', 'gap']
             timestamp  station
0  2016-01-11 10:30:00  TA00078
1  2016-01-11 10:45:00  TA00078
2  2016-01-11 11:00:00  TA00078
3  2016-01-11 11:15:00  TA00078
4  2016-01-11 11:30:00  TA00078
MEMBER 4 - EO PREPROCESSING & INTEGRATION

[PB-07] Loading EO data...

Raw EO quality:
rows                 1051200
columns                    3
stations                  40
duplicate_rows             0
missing_datetime           0
missing_radiation          0
missing_station            0
Name: EO data quality, dtype: int64

[PB-07] Detecting potential outliers...
IQR bounds: -697.42 to 1162.38
Potential outliers: 0 (0.00%)

[PB-07] Preprocessing EO data...

Clean EO shape: (1051200, 4)
Stations: 40
Remaining EO missing values: 0

[PB-